# 风险模型——以 Barra 为例完整教程：从因子暴露到协方差矩阵

## 📚 教学目标
1. 理解 **Barra 风险模型**的整体架构：因子暴露 B、因子协方差 Σ_F、特质方差 D
2. 掌握 Barra 因子体系：**国家因子 + 行业因子 + 风格因子**
3. 用矩阵运算计算 **个股风险** 和 **组合风险**
4. 理解 **纯因子投资组合** 的性质和求解方法
5. 可视化 **因子暴露热力图** 和 **风险分解**

**参考书**：《因子投资：方法与实践》（石川）第 7.2 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 场景设定：为什么需要风险模型？

### 🎯 核心问题

在因子投资中，**风险模型**的核心目标是准确估计资产的协方差矩阵 $\Sigma$：

$$\Sigma = B \Sigma_F B' + \Sigma_\varepsilon$$

其中：
- $B$ = $N \times K$ 因子暴露矩阵
- $\Sigma_F$ = $K \times K$ 因子协方差矩阵
- $\Sigma_\varepsilon$ = $N \times N$ 特质收益率协方差矩阵（对角阵）

> **降维优势**：直接估计 $N \times N$ 的 $\Sigma$ 需要估计 $N(N+1)/2$ 个参数，
> 而通过因子模型只需估计 $K(K+1)/2 + N$ 个参数，当 $K \ll N$ 时大大简化。

### 📖 Barra CNE5 模型

Barra 针对中国 A 股推出的 CNE5 模型包含：
- **1 个国家因子**：近似等于市场组合
- **P 个行业因子**：如申万一级行业
- **Q 个风格因子**：如市值、价值、动量等

模型表达式：
$$R_i = \lambda_C + \sum_{p=1}^{P} X_{ip} \lambda_{I_p} + \sum_{q=1}^{Q} S_{iq} \lambda_{S_q} + u_i$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 微型数据集：10 只股票 × 3 行业 × 2 风格因子

### 📐 Barra 模型的因子暴露矩阵

对于每只股票 $i$，因子暴露向量为：

$$\beta_i = [\underbrace{1}_{\text{国家}}, \underbrace{X_{i1}, \ldots, X_{iP}}_{\text{行业（0/1）}}, \underbrace{S_{i1}, \ldots, S_{iQ}}_{\text{风格（标准化）}}]$$

- 国家因子暴露 = 1（所有股票相同）
- 行业因子暴露 = 0 或 1（互斥）
- 风格因子暴露 = 标准化后的特征值

In [ ]:
# ========== 步骤 1: 构造微型数据集 ==========
N_STOCKS = 10
N_INDUSTRIES = 3
N_STYLE = 2  # 市值 (Size) 和 价值 (B/M)
K_FACTORS = 1 + N_INDUSTRIES + N_STYLE  # 1 + 3 + 2 = 6

stock_names = [f'S_{i+1}' for i in range(N_STOCKS)]
industry_names = ['Banking', 'Tech', 'Consumer']
style_names = ['Size', 'B/M']

# 行业归属（每只股票属于一个行业）
industry_assign = [0, 0, 0, 1, 1, 1, 1, 2, 2, 2]  # Banking:3, Tech:4, Consumer:3

# 风格因子原始值
size_raw = np.array([20.5, 21.2, 19.8, 22.1, 23.5, 21.8, 20.1, 19.5, 22.8, 21.0])
bm_raw = np.array([0.85, 1.20, 0.95, 0.45, 0.30, 0.55, 0.40, 1.10, 0.75, 1.30])

# 市值权重
market_cap = np.exp(size_raw)
cap_weights = market_cap / market_cap.sum()

print("📊 微型数据集：10 只股票")
print(f"{'股票':<6} {'行业':<12} {'Size':<10} {'B/M':<10} {'市值权重':<10}")
print("-" * 48)
for i in range(N_STOCKS):
    print(f"{stock_names[i]:<6} {industry_names[industry_assign[i]]:<12} {size_raw[i]:<10.2f} {bm_raw[i]:<10.2f} {cap_weights[i]:<10.4f}")

In [ ]:
# ========== 步骤 2: 构建因子暴露矩阵 B ==========
# B 是 N × K 矩阵

# 国家因子：所有股票暴露 = 1
country_col = np.ones(N_STOCKS)

# 行业因子：哑变量
industry_matrix = np.zeros((N_STOCKS, N_INDUSTRIES))
for i in range(N_STOCKS):
    industry_matrix[i, industry_assign[i]] = 1

# 风格因子：标准化（去均值 + 除以标准差）
# 去均值：减去市值加权均值
size_demean = size_raw - np.sum(cap_weights * size_raw)
bm_demean = bm_raw - np.sum(cap_weights * bm_raw)

# 除以标准差
size_std = size_demean / np.std(size_demean)
bm_std = bm_demean / np.std(bm_demean)

# 组合因子暴露矩阵
B = np.column_stack([country_col, industry_matrix, size_std, bm_std])

factor_names = ['Country'] + [f'Ind_{n}' for n in industry_names] + style_names

print("📊 因子暴露矩阵 B (10 × 6)")
print(f"{'股票':<6}", end='')
for fn in factor_names:
    print(f"{fn:<12}", end='')
print()
print("-" * 78)
for i in range(N_STOCKS):
    print(f"{stock_names[i]:<6}", end='')
    for j in range(K_FACTORS):
        print(f"{B[i,j]:<12.4f}", end='')
    print()

# 验证约束条件
print(f"\n📊 约束条件验证：")
print(f"  行业因子列和 = {industry_matrix.sum(axis=1)} （应全为 1）")
print(f"  市值加权风格均值 ≈ 0: Size = {np.sum(cap_weights * size_std):.6f}, B/M = {np.sum(cap_weights * bm_std):.6f}")

In [ ]:
# ========== 可视化因子暴露热力图 ==========
fig, ax = plt.subplots(figsize=(12, 6))

B_df = pd.DataFrame(B, index=stock_names, columns=factor_names)
sns.heatmap(B_df, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=1, ax=ax, cbar_kws={'label': 'Factor Exposure'})

ax.set_title('Factor Exposure Matrix B (Barra CNE5 Style)', fontsize=14, fontweight='bold')
ax.set_ylabel('Stock', fontsize=12)
ax.set_xlabel('Factor', fontsize=12)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  行业因子列：只有属于该行业的股票暴露为 1（绿色），其余为 0（黄色）。")
print(f"  风格因子列：标准化后的连续值，正负表示偏离均值的方向。")

---

## 3. 因子收益率估计（WLS 截面回归）

### 📐 加权最小二乘法

Barra 模型使用 **WLS（加权最小二乘法）** 估计因子收益率：

$$R = B \lambda + u$$

回归权重矩阵 $W$ 假设特质性收益率的方差与市值平方根成反比：

$$W = \text{diag}(\sqrt{m_1}, \sqrt{m_2}, \ldots, \sqrt{m_N})$$

加上行业因子的约束条件 $\sum_p s_p \lambda_{I_p} = 0$，使用带约束的 WLS 求解。

In [ ]:
# ========== 步骤 3: 因子收益率估计 ==========

# 模拟一期股票收益率
np.random.seed(42)

# 真实因子收益率
lambda_true = np.array([0.8,   # 国家因子（市场收益）
                        0.3,   # Banking 行业
                        -0.2,  # Tech 行业
                        0.1,   # Consumer 行业
                        0.15,  # Size 因子（小市值溢价）
                        0.25]) # B/M 因子（价值溢价）

# 生成收益率：R = B × λ + u
u = np.random.normal(0, 0.5, N_STOCKS)  # 特质收益率
R = B @ lambda_true + u

print("📊 模拟的股票收益率")
print(f"{'股票':<6} {'收益率 R':<12}")
print("-" * 18)
for i in range(N_STOCKS):
    print(f"{stock_names[i]:<6} {R[i]:<12.4f}")

# ========== WLS 求解 ==========
# 回归权重：sqrt(市值)
W = np.diag(np.sqrt(market_cap))

# 约束矩阵 C（行业因子收益率加权和为零）
# 行业权重
ind_weights = np.zeros(N_INDUSTRIES)
for p in range(N_INDUSTRIES):
    ind_weights[p] = cap_weights[industry_assign == p].sum()

# 构建约束矩阵 C: K × (K-1)
# 将第一个行业因子用其他行业因子表示
C = np.zeros((K_FACTORS, K_FACTORS - 1))
# 国家因子不受约束
C[0, 0] = 1
# 行业因子约束：λ_I1 = -(s2/s1)λ_I2 - (s3/s1)λ_I3
for j in range(1, N_INDUSTRIES):
    C[1, j] = -ind_weights[j] / ind_weights[0]  # 第一行
    C[j+1, j] = 1  # 对角线
# 风格因子不受约束
for q in range(N_STYLE):
    C[1 + N_INDUSTRIES + q, N_INDUSTRIES - 1 + q] = 1

# 纯因子组合权重矩阵 Ω = C(C'β'WβC)^{-1} C'β'W
BtWB = B.T @ W @ B
CtBtWBC = C.T @ BtWB @ C
CtBtWBC_inv = np.linalg.inv(CtBtWBC)
Omega = C @ CtBtWBC_inv @ C.T @ B.T @ W

# 因子收益率
lambda_hat = Omega @ R

# 特质收益率
u_hat = R - B @ lambda_hat

print(f"\n📊 因子收益率估计结果")
print(f"{'因子':<15} {'真实值':<12} {'估计值':<12} {'误差':<12}")
print("-" * 51)
for k in range(K_FACTORS):
    error = lambda_hat[k] - lambda_true[k]
    print(f"{factor_names[k]:<15} {lambda_true[k]:<12.4f} {lambda_hat[k]:<12.4f} {error:<12.4f}")

print(f"\n  💡 解读: WLS 估计的因子收益率与真实值非常接近。")
print(f"  约束条件验证: 行业因子加权和 = {np.sum(ind_weights * lambda_hat[1:1+N_INDUSTRIES]):.6f} (应为 0)")

---

## 4. 纯因子投资组合的性质

### 📐 纯因子投资组合

通过 WLS 求解得到的 $\Omega$ 矩阵，其每一行代表一个因子的**纯因子投资组合**权重。

纯因子投资组合满足以下性质：

1. **国家因子组合** ≈ 市场组合（多头，暴露为 1）
2. **行业因子组合**：100% 做多该行业，100% 做空国家因子组合，资金中性
3. **风格因子组合**：仅在该风格因子上有 1 单位暴露，其他因子暴露为零，资金中性

$$\Omega \cdot B = \begin{bmatrix} D_{(1+P)\times(1+P)} & 0 \\ 0 & I_{Q \times Q} \end{bmatrix}$$

In [ ]:
# ========== 步骤 4: 验证纯因子投资组合性质 ==========

# 计算 Ω × B
Omega_B = Omega @ B

print("📊 纯因子投资组合暴露矩阵 Ω × B")
print(f"{'':<15}", end='')
for fn in factor_names:
    print(f"{fn:<12}", end='')
print()
print("-" * 87)
for k in range(K_FACTORS):
    print(f"{'PF_' + factor_names[k]:<15}", end='')
    for j in range(K_FACTORS):
        print(f"{Omega_B[k,j]:<12.4f}", end='')
    print()

# 验证关键性质
print(f"\n📊 性质验证：")
print(f"  1. 国家因子组合在国家因子上暴露 = {Omega_B[0,0]:.4f} （应为 1）")
print(f"  2. 国家因子组合在行业因子上暴露 ≈ {Omega_B[0,1:1+N_INDUSTRIES]} （应接近行业权重）")
print(f"  3. 国家因子组合在风格因子上暴露 = {Omega_B[0,1+N_INDUSTRIES:]} （应为 0）")
print(f"  4. 行业因子组合资金中性: 权重和 = {[np.sum(Omega[k,:]) for k in range(1, 1+N_INDUSTRIES)]} （应为 0）")
print(f"  5. 风格因子组合资金中性: 权重和 = {[np.sum(Omega[k,:]) for k in range(1+N_INDUSTRIES, K_FACTORS)]} （应为 0）")

In [ ]:
# ========== 可视化纯因子组合权重 ==========
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for k in range(K_FACTORS):
    ax = axes[k // 3, k % 3]
    colors_pf = ['#2ecc71' if w > 0 else '#e74c3c' for w in Omega[k, :]]
    ax.bar(stock_names, Omega[k, :], color=colors_pf, edgecolor='black', alpha=0.8)
    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.set_title(f'PF: {factor_names[k]}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Weight', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Pure Factor Portfolio Weights', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  国家因子组合：所有股票权重为正（近似市值加权市场组合）。")
print(f"  行业因子组合：该行业股票权重为正，其他行业为负（多空对冲）。")
print(f"  风格因子组合：根据风格暴露分配正负权重（资金中性）。")

---

## 5. 扩展到完整规模：300 只股票 × 60 月

### 📐 协方差矩阵估计

得到因子收益率时间序列后，可以计算：

1. **因子协方差矩阵** $\hat{\Sigma}_F$：用因子收益率的时间序列计算
2. **特质方差** $\hat{\Sigma}_\varepsilon$：用特质收益率的时间序列计算（对角阵）
3. **个股协方差矩阵**：$\hat{\Sigma} = B \hat{\Sigma}_F B' + \hat{\Sigma}_\varepsilon$

In [ ]:
# ========== 步骤 5: 多期模拟 ==========
N_STOCKS = 50
N_INDUSTRIES = 5
N_STYLE = 3  # Size, B/M, Momentum
N_MONTHS = 60
K_FACTORS = 1 + N_INDUSTRIES + N_STYLE

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只")
print(f"  行业数量: {N_INDUSTRIES} 个")
print(f"  风格因子: {N_STYLE} 个")
print(f"  总因子数: {K_FACTORS} 个")
print(f"  时间跨度: {N_MONTHS} 个月")

# 随机行业归属
industry_assign = np.random.choice(N_INDUSTRIES, size=N_STOCKS)

# 模拟因子收益率时间序列
# 因子协方差矩阵（真实值）
factor_corr = np.eye(K_FACTORS)
for i in range(K_FACTORS):
    for j in range(K_FACTORS):
        if i != j:
            factor_corr[i, j] = np.random.uniform(-0.2, 0.3)
factor_corr = (factor_corr + factor_corr.T) / 2
np.fill_diagonal(factor_corr, 1.0)

factor_vol = np.array([0.05] + [0.03]*N_INDUSTRIES + [0.02]*N_STYLE)
Sigma_F_true = np.diag(factor_vol) @ factor_corr @ np.diag(factor_vol)

# 生成因子收益率序列
lambda_series = np.random.multivariate_normal(np.zeros(K_FACTORS), Sigma_F_true, N_MONTHS)

# 估计因子协方差矩阵
Sigma_F_hat = np.cov(lambda_series, rowvar=False)

print(f"\n📊 因子协方差矩阵估计")
print(f"  真实 Σ_F 对角线 (方差): {[f'{v:.6f}' for v in np.diag(Sigma_F_true)]}")
print(f"  估计 Σ_F 对角线 (方差): {[f'{v:.6f}' for v in np.diag(Sigma_F_hat)]}")

In [ ]:
# ========== 步骤 6: 计算个股协方差矩阵 ==========

# 模拟因子暴露矩阵
np.random.seed(42)
B_sim = np.zeros((N_STOCKS, K_FACTORS))
B_sim[:, 0] = 1  # 国家因子
for i in range(N_STOCKS):
    B_sim[i, 1 + industry_assign[i]] = 1  # 行业因子
B_sim[:, 1+N_INDUSTRIES:] = np.random.randn(N_STOCKS, N_STYLE) * 0.5  # 风格因子

# 特质方差（与市值成反比）
idio_vol = np.random.uniform(0.02, 0.05, N_STOCKS)
Sigma_eps = np.diag(idio_vol**2)

# 个股协方差矩阵: Σ = B Σ_F B' + Σ_ε
Sigma_stock = B_sim @ Sigma_F_hat @ B_sim.T + Sigma_eps

# 相关系数矩阵
D_stock = np.sqrt(np.diag(Sigma_stock))
corr_stock = Sigma_stock / np.outer(D_stock, D_stock)

print("📊 个股协方差矩阵估计结果")
print(f"  矩阵维度: {Sigma_stock.shape}")
print(f"  个股波动率范围: [{D_stock.min():.4f}, {D_stock.max():.4f}]")
print(f"  平均相关系数: {(corr_stock.sum() - N_STOCKS) / (N_STOCKS * (N_STOCKS - 1)):.4f}")
print(f"\n  💡 通过因子模型，只需估计 {K_FACTORS*(K_FACTORS+1)//2 + N_STOCKS} 个参数，")
print(f"     而非直接估计 {N_STOCKS*(N_STOCKS+1)//2} 个参数。")

In [ ]:
# ========== 可视化协方差矩阵 ==========
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图：因子协方差矩阵热力图
factor_labels = ['Country'] + [f'Ind_{i}' for i in range(N_INDUSTRIES)] + [f'Style_{i}' for i in range(N_STYLE)]

ax1 = axes[0]
sns.heatmap(Sigma_F_hat, annot=True, fmt='.4f', cmap='RdYlGn', center=0,
            xticklabels=factor_labels, yticklabels=factor_labels,
            linewidths=0.5, ax=ax1, cbar_kws={'label': 'Covariance'})
ax1.set_title('Factor Covariance Matrix Σ_F', fontsize=14, fontweight='bold')

# 右图：个股相关系数矩阵
ax2 = axes[1]
sns.heatmap(corr_stock, cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0, ax=ax2, cbar_kws={'label': 'Correlation'})
ax2.set_title('Stock Correlation Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：因子协方差矩阵，对角线为因子方差，非对角线为因子间协方差。")
print(f"  右图：个股相关系数矩阵，通过因子模型降维后估计的相关性结构。")

---

## 6. 偏差统计量与风险模型评估

### 📐 偏差统计量

偏差统计量用于评估风险模型的准确性：

$$b_{it} = \frac{R_{it}}{\hat{\sigma}_{it}}, \quad B_i = \sqrt{\frac{1}{T}\sum_{t=1}^T b_{it}^2}$$

如果风险预测准确，$B_i$ 应接近 1。95% 置信区间为：

$$B_i \in \left[\sqrt{1 - 1.96\sqrt{2/T}}, \sqrt{1 + 1.96\sqrt{2/T}}\right]$$

In [ ]:
# ========== 步骤 7: 偏差统计量计算 ==========

# 模拟事前风险预测和实际收益率
N_STOCKS_EVAL = 20
T_EVAL = 60

# 事前预测波动率
sigma_hat = np.random.uniform(0.03, 0.08, N_STOCKS_EVAL)

# 模拟收益率（使用预测波动率 + 噪声）
R_eval = np.zeros((T_EVAL, N_STOCKS_EVAL))
for t in range(T_EVAL):
    R_eval[t, :] = sigma_hat * np.random.randn(N_STOCKS_EVAL)

# 计算标准化收益率 b_it
b = R_eval / sigma_hat  # T × N

# 偏差统计量 B_i
B_stat = np.sqrt(np.mean(b**2, axis=0))

# 95% 置信区间
ci_lower = np.sqrt(1 - 1.96 * np.sqrt(2 / T_EVAL))
ci_upper = np.sqrt(1 + 1.96 * np.sqrt(2 / T_EVAL))

print("📊 偏差统计量评估")
print(f"  样本量: {N_STOCKS_EVAL} 只股票 × {T_EVAL} 期")
print(f"  95% 置信区间: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"\n  B_i 范围: [{B_stat.min():.4f}, {B_stat.max():.4f}]")
print(f"  B_i 均值: {B_stat.mean():.4f}")

n_outside = np.sum((B_stat < ci_lower) | (B_stat > ci_upper))
print(f"  超出置信区间的股票数: {n_outside} / {N_STOCKS_EVAL}")

if abs(B_stat.mean() - 1) < 0.1:
    print(f"\n  💡 偏差统计量均值接近 1，风险模型预测较准确！")
else:
    print(f"\n  💡 偏差统计量偏离 1，风险模型需要调整。")

In [ ]:
# ========== 可视化偏差统计量 ==========
fig, ax = plt.subplots(figsize=(10, 6))

colors_bias = ['#2ecc71' if ci_lower <= b <= ci_upper else '#e74c3c' for b in B_stat]
ax.bar(range(N_STOCKS_EVAL), B_stat, color=colors_bias, edgecolor='black', alpha=0.8)
ax.axhline(y=1, color='blue', linewidth=2, linestyle='--', label='Ideal B = 1')
ax.axhline(y=ci_lower, color='orange', linewidth=1.5, linestyle=':', label=f'95% CI [{ci_lower:.3f}, {ci_upper:.3f}]')
ax.axhline(y=ci_upper, color='orange', linewidth=1.5, linestyle=':')

ax.set_xlabel('Stock Index', fontsize=12)
ax.set_ylabel('Bias Statistic B_i', fontsize=12)
ax.set_title('Risk Model Accuracy: Bias Statistics', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  绿色柱：B_i 在 95% 置信区间内，风险预测准确。")
print(f"  红色柱：B_i 超出置信区间，风险预测有偏。")
print(f"  蓝色虚线为理想值 1，橙色虚线为置信区间边界。")

---

## 7. 组合风险分解

### 📐 风险分解

给定组合权重 $w$，组合方差可以分解为因子风险和特质风险：

$$\sigma_p^2 = w' \Sigma w = (w'B) \Sigma_F (w'B)' + w' \Sigma_\varepsilon w$$

其中：
- $w'B$ = 组合在各因子上的暴露
- $(w'B) \Sigma_F (w'B)'$ = 因子风险贡献
- $w' \Sigma_\varepsilon w$ = 特质风险贡献

In [ ]:
# ========== 步骤 8: 组合风险分解 ==========

# 假设等权组合
w_equal = np.ones(N_STOCKS) / N_STOCKS

# 组合因子暴露
port_factor_exposure = w_equal @ B_sim  # K 维向量

# 因子风险
factor_risk = port_factor_exposure @ Sigma_F_hat @ port_factor_exposure

# 特质风险
idio_risk = w_equal @ Sigma_eps @ w_equal

# 总风险
total_risk = factor_risk + idio_risk

print("📊 等权组合风险分解")
print(f"  总风险 (组合方差)  = {total_risk:.6f}")
print(f"  因子风险           = {factor_risk:.6f} ({factor_risk/total_risk*100:.1f}%)")
print(f"  特质风险           = {idio_risk:.6f} ({idio_risk/total_risk*100:.1f}%)")
print(f"  组合波动率         = {np.sqrt(total_risk):.4f}")

# 各因子风险贡献
print(f"\n📊 各因子风险贡献")
print(f"{'因子':<15} {'暴露':<12} {'风险贡献':<12} {'占比':<10}")
print("-" * 49)
for k in range(K_FACTORS):
    contrib_k = port_factor_exposure[k]**2 * Sigma_F_hat[k, k]
    # 加上交叉项
    for j in range(K_FACTORS):
        if j != k:
            contrib_k += 0.5 * port_factor_exposure[k] * port_factor_exposure[j] * Sigma_F_hat[k, j]
    print(f"{factor_labels[k]:<15} {port_factor_exposure[k]:<12.4f} {contrib_k:<12.6f} {contrib_k/factor_risk*100:<10.1f}%")

In [ ]:
# ========== 可视化风险分解 ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：因子风险 vs 特质风险（饼图）
ax1 = axes[0]
sizes = [factor_risk, idio_risk]
labels_pie = [f'Factor Risk\n({factor_risk/total_risk*100:.1f}%)',
              f'Idiosyncratic Risk\n({idio_risk/total_risk*100:.1f}%)']
colors_pie = ['#3498db', '#e74c3c']
explode = (0.05, 0.05)
ax1.pie(sizes, explode=explode, labels=labels_pie, colors=colors_pie,
        autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 11})
ax1.set_title('Portfolio Risk Decomposition', fontsize=14, fontweight='bold')

# 右图：各因子风险贡献
ax2 = axes[1]
factor_contribs = []
for k in range(K_FACTORS):
    contrib_k = port_factor_exposure[k]**2 * Sigma_F_hat[k, k]
    for j in range(K_FACTORS):
        if j != k:
            contrib_k += 0.5 * port_factor_exposure[k] * port_factor_exposure[j] * Sigma_F_hat[k, j]
    factor_contribs.append(contrib_k)

colors_fc = plt.cm.Set3(np.linspace(0, 1, K_FACTORS))
bars = ax2.barh(factor_labels, factor_contribs, color=colors_fc, edgecolor='black', alpha=0.8)
ax2.set_xlabel('Risk Contribution', fontsize=12)
ax2.set_title('Factor Risk Contributions', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：组合风险中因子风险占主导，特质风险因分散化而降低。")
print(f"  右图：各因子对组合因子风险的贡献，国家因子通常贡献最大。")

---

## 📌 核心概念回顾

### 📌 Barra 风险模型架构
- **定义**: 通过多因子模型实现降维，方便估计资产协方差矩阵
- **公式**: $\Sigma = B \Sigma_F B' + \Sigma_\varepsilon$
- **优势**: 参数数量从 $N(N+1)/2$ 降至 $K(K+1)/2 + N$

### 📌 因子体系
- **国家因子**: 暴露为 1，近似市场组合
- **行业因子**: 哑变量（0/1），资金中性
- **风格因子**: 标准化后的特征值，资金中性

### 📌 纯因子投资组合
- **定义**: 通过 WLS 求解得到的因子投资组合
- **性质**: 行业/风格组合资金中性，仅在目标因子上有暴露
- **公式**: $\Omega = C(C'\beta'W\beta C)^{-1} C'\beta'W$

### 📌 偏差统计量
- **定义**: $B_i = \sqrt{\frac{1}{T}\sum_t (R_{it}/\hat{\sigma}_{it})^2}$
- **含义**: 衡量风险预测的准确性
- **判断标准**: $B_i$ 应接近 1，超出置信区间表示预测有偏

### 🔗 完整流程
```
因子暴露 B → 截面回归(WLS) → 因子收益率 λ_t
      ↓              ↓              ↓
  行业哑变量     纯因子组合 Ω    时序积累
  风格标准化         ↓              ↓
               因子协方差 Σ_F ← 因子收益率序列
                    ↓
           个股协方差 Σ = BΣ_FB' + Σ_ε
                    ↓
              组合优化 / 风险管理
```

### 📝 关键公式汇总

| 公式 | 含义 |
|------|------|
| $\Sigma = B\Sigma_F B' + \Sigma_\varepsilon$ | 个股协方差矩阵分解 |
| $R = B\lambda + u$ | Barra 模型表达式 |
| $\Omega = C(C'\beta'W\beta C)^{-1}C'\beta'W$ | 纯因子组合权重 |
| $B_i = \sqrt{\frac{1}{T}\sum b_{it}^2}$ | 偏差统计量 |
| $\sigma_p^2 = (w'B)\Sigma_F(w'B)' + w'\Sigma_\varepsilon w$ | 组合风险分解 |

## ❌ 常见误区

### ❌ 误区 1: 直接用历史收益率估计协方差矩阵
**✓ 正确理解**: 直接估计 $N \times N$ 协方差矩阵需要估计 $N(N+1)/2$ 个参数，当 $N$ 很大时估计误差极大。通过因子模型降维，只需估计 $K(K+1)/2 + N$ 个参数，大大提升估计精度。

### ❌ 误区 2: 因子暴露必须用回归系数
**✓ 正确理解**: Barra 模型直接使用公司特征值（如 BM、市值的对数）作为因子暴露的原始值，再进行标准化处理。这与使用时序回归系数的方法不同，但更直观且信息更及时。

### ❌ 误区 3: 行业因子收益率没有约束
**✓ 正确理解**: 国家因子暴露和行业因子暴露存在共线性（所有股票国家暴露为 1，行业暴露之和也为 1）。因此需要约束行业因子收益率的市值加权和为零：$\sum_p s_p \lambda_{I_p} = 0$。

### ❌ 误区 4: 纯因子组合就是等权多空组合
**✓ 正确理解**: 纯因子投资组合通过 WLS 优化求解，其权重使得组合仅在目标因子上有暴露，其他因子暴露为零。等权多空组合无法满足这一性质。

### ❌ 误区 5: 协方差矩阵估计不需要调整
**✓ 正确理解**: 历史样本协方差矩阵存在大量估计误差。Barra 使用特征因子调整法（调整 $\hat{\Sigma}_F$）和贝叶斯收缩法（调整 $\hat{\Sigma}_\varepsilon$）来提升估计精度，以降低偏差统计量为目标。